In [1]:
# ============================================================
# Notebook 11 — Rolling Reforecast Evaluation
# Cell 1 — Environment setup
# ============================================================

from pathlib import Path
import warnings
import random
import time

import numpy as np
import pandas as pd

from lightgbm import LGBMRegressor

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

DATA_FOLDER = Path("../data/favorita")
CLEAN_DATA_PATH = DATA_FOLDER / "benchmark_300_cleaned.csv"

# ------------------------------------------------------------
# Rolling reforecast setup
# ------------------------------------------------------------

TRAIN_END = pd.Timestamp("2017-04-30")

FIRST_FORECAST_START = pd.Timestamp("2017-05-01")
FIRST_FORECAST_DAYS = 30

SECOND_FORECAST_START = pd.Timestamp("2017-05-31")
SECOND_FORECAST_DAYS = 30

print("Notebook 11 ready.")
print("Train through:", TRAIN_END.date())
print("First forecast:", FIRST_FORECAST_START.date(), "for", FIRST_FORECAST_DAYS, "days")
print("Second forecast:", SECOND_FORECAST_START.date(), "for", SECOND_FORECAST_DAYS, "days")

Notebook 11 ready.
Train through: 2017-04-30
First forecast: 2017-05-01 for 30 days
Second forecast: 2017-05-31 for 30 days


In [2]:
# ============================================================
# Cell 2 — Load cleaned benchmark data and metadata
# ============================================================

data = pd.read_csv(
    CLEAN_DATA_PATH,
    parse_dates=["date"]
)

data = (
    data
    .sort_values(["store_nbr", "item_nbr", "date"])
    .reset_index(drop=True)
)

# Static metadata
items = pd.read_csv(DATA_FOLDER / "items.csv")[
    ["item_nbr", "family", "class", "perishable"]
]

stores = pd.read_csv(DATA_FOLDER / "stores.csv")[
    ["store_nbr", "city", "state", "type", "cluster"]
].rename(columns={"type": "store_type"})

data = (
    data
    .merge(items, on="item_nbr", how="left")
    .merge(stores, on="store_nbr", how="left")
)

print("Shape:", data.shape)
print(
    "Series:",
    data[["store_nbr", "item_nbr"]]
    .drop_duplicates()
    .shape[0]
)
print("Date range:", data["date"].min(), "to", data["date"].max())
print("Missing targets:", data["demand_target"].isna().sum())

Shape: (480213, 28)
Series: 300
Date range: 2013-01-01 00:00:00 to 2017-08-15 00:00:00
Missing targets: 4458


In [3]:
# ============================================================
# Cell 3 — Rebuild final LightGBM features
# ============================================================

data = (
    data
    .sort_values(["store_nbr", "item_nbr", "date"])
    .reset_index(drop=True)
)

grouped = data.groupby(
    ["store_nbr", "item_nbr"],
    observed=True
)["demand_target"]

# ------------------------------------------------------------
# Lag features
# ------------------------------------------------------------

for lag in [1, 7, 14, 21, 28, 35, 42, 56]:
    data[f"lag_{lag}"] = grouped.shift(lag)

# ------------------------------------------------------------
# Rolling features
# IMPORTANT: shift first so current/future demand is not leaked
# ------------------------------------------------------------

previous_demand = grouped.shift(1)

group_keys = [
    data["store_nbr"],
    data["item_nbr"],
]

for window in [7, 14, 28, 56]:

    rolling = previous_demand.groupby(
        group_keys,
        observed=True
    ).rolling(
        window,
        min_periods=1
    )

    data[f"rolling_mean_{window}"] = (
        rolling.mean()
        .reset_index(level=[0, 1], drop=True)
    )

for window in [7, 28]:

    rolling = previous_demand.groupby(
        group_keys,
        observed=True
    ).rolling(
        window,
        min_periods=1
    )

    data[f"rolling_std_{window}"] = (
        rolling.std()
        .reset_index(level=[0, 1], drop=True)
    )

    data[f"rolling_median_{window}"] = (
        rolling.median()
        .reset_index(level=[0, 1], drop=True)
    )

# ------------------------------------------------------------
# Zero-rate feature
# ------------------------------------------------------------

data["zero_rate_28"] = (
    previous_demand.eq(0)
    .groupby(group_keys, observed=True)
    .rolling(28, min_periods=1)
    .mean()
    .reset_index(level=[0, 1], drop=True)
)

# ------------------------------------------------------------
# Calendar features
# ------------------------------------------------------------

data["day_of_week"] = data["date"].dt.dayofweek
data["month"] = data["date"].dt.month

# ------------------------------------------------------------
# Feature lists
# ------------------------------------------------------------

STATIC_FEATURES = [
    "store_nbr",
    "item_nbr",
    "family",
    "class",
    "perishable",
    "city",
    "state",
    "store_type",
    "cluster",
]

LAG_FEATURES = [
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_21",
    "lag_28",
    "lag_35",
    "lag_42",
    "lag_56",
]

ROLLING_FEATURES = [
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28",
    "rolling_mean_56",
    "rolling_std_7",
    "rolling_std_28",
    "rolling_median_7",
    "rolling_median_28",
    "zero_rate_28",
]

TIME_FEATURES = [
    "day_of_week",
    "month",
    "is_holiday",
    "is_event",
    "is_additional",
    "is_transfer",
    "is_bridge",
    "is_work_day",
]

BUSINESS_FEATURES = [
    "promotion_status",
]

FEATURE_COLUMNS = (
    STATIC_FEATURES
    + LAG_FEATURES
    + ROLLING_FEATURES
    + TIME_FEATURES
    + BUSINESS_FEATURES
)

CATEGORICAL_FEATURES = [
    "store_nbr",
    "item_nbr",
    "family",
    "class",
    "city",
    "state",
    "store_type",
    "cluster",
    "day_of_week",
    "month",
    "promotion_status",
]

for col in CATEGORICAL_FEATURES:
    data[col] = data[col].astype("category")

print("Feature count:", len(FEATURE_COLUMNS))
print("Categorical count:", len(CATEGORICAL_FEATURES))
print("Feature rebuild complete.")

Feature count: 35
Categorical count: 11
Feature rebuild complete.


In [4]:
# ============================================================
# Cell 3B — Restore missing frozen LightGBM features
# ============================================================

# Continuous time index within each store-item series
data["time_idx"] = (
    data.groupby(
        ["store_nbr", "item_nbr"],
        observed=True
    ).cumcount()
)

# Number of available historical observations in each window
previous_demand = data.groupby(
    ["store_nbr", "item_nbr"],
    observed=True
)["demand_target"].shift(1)

group_keys = [
    data["store_nbr"],
    data["item_nbr"],
]

for window in [7, 14, 28, 56]:

    data[f"rolling_count_{window}"] = (
        previous_demand
        .groupby(group_keys, observed=True)
        .rolling(window, min_periods=1)
        .count()
        .reset_index(level=[0, 1], drop=True)
    )

# Add them to the exact frozen feature set
ROLLING_FEATURES = [
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28",
    "rolling_mean_56",

    "rolling_count_7",
    "rolling_count_14",
    "rolling_count_28",
    "rolling_count_56",

    "rolling_std_7",
    "rolling_std_28",
    "rolling_median_7",
    "rolling_median_28",
    "zero_rate_28",
]

TIME_FEATURES = [
    "day_of_week",
    "month",
    "time_idx",
    "is_holiday",
    "is_event",
    "is_additional",
    "is_transfer",
    "is_bridge",
    "is_work_day",
]

FEATURE_COLUMNS = (
    STATIC_FEATURES
    + LAG_FEATURES
    + ROLLING_FEATURES
    + TIME_FEATURES
    + BUSINESS_FEATURES
)

print("Corrected feature count:", len(FEATURE_COLUMNS))
print("Expected frozen feature count: 40")

Corrected feature count: 40
Expected frozen feature count: 40


In [5]:
# ============================================================
# Cell 4 — Train frozen Optuna LightGBM
# ============================================================

train_data = data[
    (data["date"] <= TRAIN_END)
    & data["demand_target"].notna()
].copy()

OPTUNA_PARAMS = {
    "objective": "regression_l1",
    "learning_rate": 0.01082401838150096,
    "num_leaves": 117,
    "min_child_samples": 46,
    "max_depth": 16,
    "reg_alpha": 7.7513282336111455,
    "reg_lambda": 9.394989415641891,
    "colsample_bytree": 0.9684482051282947,
    "subsample": 0.8793699936433255,
    "subsample_freq": 1,
    "min_split_gain": 0.4609371175115584,
    "random_state": SEED,
    "n_jobs": -1,
    "verbosity": -1,
}

# BT3's tuned training used 1816 trees.
rolling_model = LGBMRegressor(
    **OPTUNA_PARAMS,
    n_estimators=1816,
)

start_time = time.time()

rolling_model.fit(
    train_data[FEATURE_COLUMNS],
    train_data["demand_target"],
    categorical_feature=CATEGORICAL_FEATURES,
)

print("Training rows:", f"{len(train_data):,}")
print(
    "Training time:",
    round((time.time() - start_time) / 60, 2),
    "minutes"
)
print("Frozen LightGBM trained.")

Training rows: 443,774
Training time: 0.72 minutes
Frozen LightGBM trained.


In [6]:
# ============================================================
# Cell 5 — Recursive forecasting helper + first 30-day forecast
# ============================================================

def rolling_recursive_forecast(
    model,
    full_data,
    history_end,
    forecast_start,
    horizon_days,
):
    """
    Forecast one day at a time.

    Only demand available through history_end is treated as actual history.
    During the forecast window, previous model predictions are used as
    temporary history.
    """

    # --------------------------------------------------------
    # Working demand history
    # --------------------------------------------------------

    working = full_data.copy()

    # Hide all demand after the history cutoff.
    working.loc[
        working["date"] > history_end,
        "working_demand"
    ] = np.nan

    working.loc[
        working["date"] <= history_end,
        "working_demand"
    ] = working.loc[
        working["date"] <= history_end,
        "demand_target"
    ]

    predictions = []

    # --------------------------------------------------------
    # Predict each future day sequentially
    # --------------------------------------------------------

    for step in range(horizon_days):

        forecast_date = (
            forecast_start
            + pd.Timedelta(days=step)
        )

        day_rows = working[
            working["date"] == forecast_date
        ].copy()

        feature_rows = []

        # Build history-based features separately
        # for each of the 300 store-item series.
        for idx, row in day_rows.iterrows():

            store = row["store_nbr"]
            item = row["item_nbr"]

            history = working[
                (working["store_nbr"] == store)
                & (working["item_nbr"] == item)
                & (working["date"] < forecast_date)
            ]["working_demand"]

            values = history.to_numpy()

            # ----------------------------
            # Lag features
            # ----------------------------

            for lag in [1, 7, 14, 21, 28, 35, 42, 56]:

                if len(values) >= lag:
                    row[f"lag_{lag}"] = values[-lag]
                else:
                    row[f"lag_{lag}"] = np.nan

            # ----------------------------
            # Rolling features
            # ----------------------------

            for window in [7, 14, 28, 56]:

                recent = pd.Series(
                    values[-window:]
                )

                row[f"rolling_mean_{window}"] = (
                    recent.mean()
                )

                row[f"rolling_count_{window}"] = (
                    recent.count()
                )

            for window in [7, 28]:

                recent = pd.Series(
                    values[-window:]
                )

                row[f"rolling_std_{window}"] = (
                    recent.std()
                )

                row[f"rolling_median_{window}"] = (
                    recent.median()
                )

            # Same zero-rate definition used in our
            # LightGBM feature pipeline.
            recent_28 = pd.Series(
                values[-28:]
            )

            row["zero_rate_28"] = (
                recent_28.eq(0).mean()
            )

            feature_rows.append(row)

        forecast_features = pd.DataFrame(feature_rows)

        # Restore the categorical dtypes expected by LightGBM.
        for col in CATEGORICAL_FEATURES:

            forecast_features[col] = pd.Categorical(
                forecast_features[col],
                categories=data[col].cat.categories
            )

        # Predict today's demand.
        day_predictions = np.clip(
            model.predict(
                forecast_features[FEATURE_COLUMNS]
            ),
            0,
            None
        )

        forecast_features["prediction"] = day_predictions

        # ----------------------------------------------------
        # Feed predictions into temporary history
        # ----------------------------------------------------

        for (_, predicted_row), prediction in zip(
            forecast_features.iterrows(),
            day_predictions
        ):

            mask = (
                (working["store_nbr"] == predicted_row["store_nbr"])
                & (working["item_nbr"] == predicted_row["item_nbr"])
                & (working["date"] == forecast_date)
            )

            working.loc[
                mask,
                "working_demand"
            ] = prediction

        predictions.append(
            forecast_features[
                [
                    "store_nbr",
                    "item_nbr",
                    "date",
                    "prediction",
                ]
            ].copy()
        )

    return pd.concat(
        predictions,
        ignore_index=True
    )


# ============================================================
# FIRST 30 DAYS: MAY 1 → MAY 30
# ============================================================

start = time.time()

may_forecast = rolling_recursive_forecast(
    model=rolling_model,
    full_data=data,
    history_end=TRAIN_END,
    forecast_start=FIRST_FORECAST_START,
    horizon_days=FIRST_FORECAST_DAYS,
)

print("Forecast rows:", len(may_forecast))
print(
    "Forecast range:",
    may_forecast["date"].min(),
    "to",
    may_forecast["date"].max()
)

print(
    "Runtime:",
    round((time.time() - start) / 60, 2),
    "minutes"
)

Forecast rows: 9000
Forecast range: 2017-05-01 00:00:00 to 2017-05-30 00:00:00
Runtime: 1.2 minutes


In [7]:
# ============================================================
# Cell 6 — Second 30-day forecast with refreshed ACTUAL history
# ============================================================

# For this forecast, actual demand through May 30 is now available.
UPDATED_HISTORY_END = pd.Timestamp("2017-05-30")

start = time.time()

june_forecast = rolling_recursive_forecast(
    model=rolling_model,
    full_data=data,
    history_end=UPDATED_HISTORY_END,
    forecast_start=SECOND_FORECAST_START,
    horizon_days=SECOND_FORECAST_DAYS,
)

print("Forecast rows:", len(june_forecast))

print(
    "Forecast range:",
    june_forecast["date"].min(),
    "to",
    june_forecast["date"].max()
)

print(
    "Runtime:",
    round((time.time() - start) / 60, 2),
    "minutes"
)

Forecast rows: 9000
Forecast range: 2017-05-31 00:00:00 to 2017-06-29 00:00:00
Runtime: 1.17 minutes


In [8]:
# ============================================================
# Cell 7 — Combine both rolling forecasts and evaluate
# ============================================================

rolling_forecast = pd.concat(
    [
        may_forecast,
        june_forecast,
    ],
    ignore_index=True
)

# Attach actual demand
actuals = data[
    [
        "store_nbr",
        "item_nbr",
        "date",
        "demand_target",
    ]
].rename(
    columns={
        "demand_target": "actual"
    }
)

rolling_eval = rolling_forecast.merge(
    actuals,
    on=["store_nbr", "item_nbr", "date"],
    how="left",
)

# Evaluate only rows where actual demand is known
rolling_eval = rolling_eval[
    rolling_eval["actual"].notna()
].copy()

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

rolling_mae = np.mean(
    np.abs(
        rolling_eval["actual"]
        - rolling_eval["prediction"]
    )
)

rolling_wape = (
    np.abs(
        rolling_eval["actual"]
        - rolling_eval["prediction"]
    ).sum()
    / np.abs(
        rolling_eval["actual"]
    ).sum()
    * 100
)

print("Rows evaluated:", len(rolling_eval))

print(
    "Rolling 30+30 MAE:",
    round(rolling_mae, 4)
)

print(
    "Rolling 30+30 WAPE:",
    round(rolling_wape, 4),
    "%"
)

print("\nOld fixed 60-day BT3:")
print("MAE:  3.1676")
print("WAPE: 43.4885 %")

print("\nImprovement in WAPE:")
print(
    round(
        43.4885 - rolling_wape,
        4
    ),
    "percentage points"
)

Rows evaluated: 17998
Rolling 30+30 MAE: 3.1043
Rolling 30+30 WAPE: 42.6187 %

Old fixed 60-day BT3:
MAE:  3.1676
WAPE: 43.4885 %

Improvement in WAPE:
0.8698 percentage points


In [9]:
# ============================================================
# Cell 8 — Retrain model after actual May sales arrive
# ============================================================

UPDATED_HISTORY_END = pd.Timestamp("2017-05-30")

updated_train_data = data[
    (data["date"] <= UPDATED_HISTORY_END)
    & data["demand_target"].notna()
].copy()

updated_model = LGBMRegressor(
    **OPTUNA_PARAMS,
    n_estimators=1816,
)

start = time.time()

updated_model.fit(
    updated_train_data[FEATURE_COLUMNS],
    updated_train_data["demand_target"],
    categorical_feature=CATEGORICAL_FEATURES,
)

print("Updated training rows:", f"{len(updated_train_data):,}")

print(
    "Retraining time:",
    round((time.time() - start) / 60, 2),
    "minutes"
)

print("LightGBM retrained through May 30.")

Updated training rows: 452,774
Retraining time: 0.91 minutes
LightGBM retrained through May 30.


In [10]:
# ============================================================
# Cell 9 — Second 30-day forecast using RETRAINED LightGBM
# ============================================================

start = time.time()

june_forecast_retrained = rolling_recursive_forecast(
    model=updated_model,
    full_data=data,
    history_end=UPDATED_HISTORY_END,   # actual data through May 30
    forecast_start=SECOND_FORECAST_START,
    horizon_days=SECOND_FORECAST_DAYS,
)

print("Forecast rows:", len(june_forecast_retrained))

print(
    "Forecast range:",
    june_forecast_retrained["date"].min(),
    "to",
    june_forecast_retrained["date"].max()
)

print(
    "Runtime:",
    round((time.time() - start) / 60, 2),
    "minutes"
)

Forecast rows: 9000
Forecast range: 2017-05-31 00:00:00 to 2017-06-29 00:00:00
Runtime: 1.47 minutes


In [12]:
# ============================================================
# Cell 10 — Evaluate full rolling + retraining strategy
# ============================================================

# First 30 days came from the original model.
# Second 30 days came from the retrained model.
rolling_retrained_forecast = pd.concat(
    [
        may_forecast,
        june_forecast_retrained,
    ],
    ignore_index=True
)

actuals = data[
    [
        "store_nbr",
        "item_nbr",
        "date",
        "demand_target",
    ]
].rename(
    columns={"demand_target": "actual"}
)

rolling_retrained_eval = rolling_retrained_forecast.merge(
    actuals,
    on=["store_nbr", "item_nbr", "date"],
    how="left",
)

rolling_retrained_eval = rolling_retrained_eval[
    rolling_retrained_eval["actual"].notna()
].copy()

# MAE
retrained_mae = np.mean(
    np.abs(
        rolling_retrained_eval["actual"]
        - rolling_retrained_eval["prediction"]
    )
)

# WAPE
retrained_wape = (
    np.abs(
        rolling_retrained_eval["actual"]
        - rolling_retrained_eval["prediction"]
    ).sum()
    /
    np.abs(
        rolling_retrained_eval["actual"]
    ).sum()
    * 100
)

print("Rows evaluated:", len(rolling_retrained_eval))

print(
    "\nRolling + retraining MAE:",
    round(retrained_mae, 4)
)

print(
    "Rolling + retraining WAPE:",
    round(retrained_wape, 4),
    "%"
)

print("\n==============================")
print("LIGHTGBM COMPARISON")
print("==============================")

print("Fixed 60-day:")
print("  MAE  = 3.1676")
print("  WAPE = 43.4885%")

print("\n30+30, refreshed actual history only:")
print("  MAE  = 3.1043")
print("  WAPE = 42.6187%")

print("\n30+30, refreshed history + retraining:")
print("  MAE  =", round(retrained_mae, 4))
print("  WAPE =", round(retrained_wape, 4), "%")

print(
    "\nWAPE improvement vs fixed 60-day:",
    round(43.4885 - retrained_wape, 4),
    "percentage points"
)

Rows evaluated: 17998

Rolling + retraining MAE: 3.1051
Rolling + retraining WAPE: 42.63 %

LIGHTGBM COMPARISON
Fixed 60-day:
  MAE  = 3.1676
  WAPE = 43.4885%

30+30, refreshed actual history only:
  MAE  = 3.1043
  WAPE = 42.6187%

30+30, refreshed history + retraining:
  MAE  = 3.1051
  WAPE = 42.63 %

WAPE improvement vs fixed 60-day: 0.8585 percentage points


In [13]:
# ============================================================
# Cell 11 — Load saved global TFT and its artifacts
# ============================================================

from pathlib import Path
import pickle
import json

import torch

from pytorch_forecasting import (
    TimeSeriesDataSet,
    TemporalFusionTransformer,
)

TFT_FOLDER = Path("../models/tft")

TFT_MODEL_PATH = (
    TFT_FOLDER / "final_global_tft.ckpt"
)

TFT_DATA_PATH = (
    DATA_FOLDER / "tft_300_prepared.pkl"
)

TFT_PARAMETERS_PATH = (
    TFT_FOLDER / "tft_dataset_parameters.pkl"
)

TFT_CONFIG_PATH = (
    TFT_FOLDER / "tft_experiment_config.json"
)

# ------------------------------------------------------------
# Load trained TFT
# ------------------------------------------------------------

tft_model = TemporalFusionTransformer.load_from_checkpoint(
    TFT_MODEL_PATH,
    map_location="cpu"
)

tft_model.eval()

# ------------------------------------------------------------
# Load prepared data
# ------------------------------------------------------------

tft_data = pd.read_pickle(
    TFT_DATA_PATH
)

# ------------------------------------------------------------
# Load fitted dataset parameters
# Includes categorical encoders, scalers, normalizers, etc.
# ------------------------------------------------------------

with open(TFT_PARAMETERS_PATH, "rb") as f:
    tft_dataset_parameters = pickle.load(f)

# Experiment information
with open(TFT_CONFIG_PATH, "r") as f:
    tft_config = json.load(f)

print("TFT loaded successfully.")
print("Rows:", len(tft_data))
print("Series:", tft_data["series_id"].nunique())
print("Model prediction length:", tft_config["max_prediction_length"])
print("Saved validation loss:", round(tft_config["best_validation_loss"], 4))

TFT loaded successfully.
Rows: 480213
Series: 300
Model prediction length: 60
Saved validation loss: 2.0258


In [14]:
# ============================================================
# Cell 12 — TFT forecast from Apr 30 history
#            May 1 → June 29 prediction window
# ============================================================

FIRST_TFT_CUTOFF = pd.Timestamp("2017-06-29")

# Keep data only through the end of the 60-day forecast window.
# This makes the final prediction window:
#
# Encoder history → through Apr 30
# Decoder         → May 1 through Jun 29
first_tft_data = tft_data[
    tft_data["date"] <= FIRST_TFT_CUTOFF
].copy()

# Rebuild a prediction dataset using the EXACT encoders,
# scalers and configuration saved during TFT training.
first_tft_dataset = TimeSeriesDataSet.from_parameters(
    tft_dataset_parameters,
    first_tft_data,
    predict=True,
    stop_randomization=True,
)

first_tft_loader = first_tft_dataset.to_dataloader(
    train=False,
    batch_size=256,
    num_workers=0,
)

print("Prediction sequences:", len(first_tft_dataset))
print("Prediction batches:", len(first_tft_loader))

Prediction sequences: 300
Prediction batches: 2


In [15]:
# ============================================================
# Cell 13 — Generate TFT quantile predictions
# ============================================================

with torch.no_grad():
    tft_output = tft_model.predict(
        first_tft_loader,
        mode="quantiles",
        return_x=True
    )

# Quantile predictions:
# shape should be approximately:
# [300 series, 60 forecast days, 3 quantiles]
quantile_predictions = tft_output.output.cpu().numpy()

print("Prediction shape:", quantile_predictions.shape)

# P10, P50, P90
p10 = quantile_predictions[:, :, 0]
p50 = quantile_predictions[:, :, 1]
p90 = quantile_predictions[:, :, 2]

print("P10 shape:", p10.shape)
print("P50 shape:", p50.shape)
print("P90 shape:", p90.shape)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


Prediction shape: (300, 60, 3)
P10 shape: (300, 60)
P50 shape: (300, 60)
P90 shape: (300, 60)


In [16]:
# ============================================================
# Cell 14 — Inspect TFT prediction indexing
# ============================================================

print("Output object type:", type(tft_output))

print("\nAvailable fields:")
print(tft_output._fields)

print("\nX keys:")
print(tft_output.x.keys())

print("\nGroups shape:")
print(tft_output.x["groups"].shape)

print("\nDecoder time_idx shape:")
print(tft_output.x["decoder_time_idx"].shape)

print("\nFirst 5 group IDs:")
print(tft_output.x["groups"][:5])

print("\nFirst forecast time_idx values:")
print(tft_output.x["decoder_time_idx"][0, :10])

Output object type: <class 'pytorch_forecasting.models.base._base_model.Prediction'>

Available fields:
('output', 'x', 'index', 'decoder_lengths', 'y')

X keys:
dict_keys(['encoder_cat', 'encoder_cont', 'encoder_target', 'encoder_lengths', 'decoder_cat', 'decoder_cont', 'decoder_target', 'decoder_lengths', 'decoder_time_idx', 'groups', 'target_scale'])

Groups shape:
torch.Size([300, 1])

Decoder time_idx shape:
torch.Size([300, 60])

First 5 group IDs:
tensor([[0],
        [1],
        [2],
        [3],
        [4]])

First forecast time_idx values:
tensor([1580, 1581, 1582, 1583, 1584, 1585, 1586, 1587, 1588, 1589])


In [18]:
# ============================================================
# Cell 15 — Recover TFT group-code → series_id mapping
# ============================================================

# PyTorch Forecasting stores the categorical encoder
# for the group id inside the dataset parameters.
series_encoder = (
    tft_dataset_parameters["categorical_encoders"]["__group_id__series_id"]
)

# Encoded integer -> original series_id
group_code_to_series = {
    code: series
    for series, code in series_encoder.classes_.items()
}

print("Number of mapped series:", len(group_code_to_series))

print("\nFirst 10 mappings:")
for code in range(10):
    print(code, "->", group_code_to_series[code])

# Verify prediction group codes
group_codes = (
    tft_output.x["groups"]
    .squeeze(1)
    .cpu()
    .numpy()
)

print("\nPrediction group count:", len(group_codes))
print("First 10 prediction group codes:", group_codes[:10])

Number of mapped series: 300

First 10 mappings:
0 -> 10_1080001
1 -> 10_1087467
2 -> 10_1102971
3 -> 10_1464198
4 -> 10_214381
5 -> 10_409904
6 -> 10_648313
7 -> 10_881912
8 -> 10_964753
9 -> 11_1239815

Prediction group count: 300
First 10 prediction group codes: [0 1 2 3 4 5 6 7 8 9]


In [19]:
# ============================================================
# Cell 16 — Build TFT prediction table
# ============================================================

forecast_rows = []

decoder_time_idx = (
    tft_output.x["decoder_time_idx"]
    .cpu()
    .numpy()
)

for i, group_code in enumerate(group_codes):

    series_id = group_code_to_series[int(group_code)]

    series_dates = (
        tft_data[
            tft_data["series_id"] == series_id
        ][["time_idx", "date", "store_nbr", "item_nbr"]]
        .drop_duplicates()
        .set_index("time_idx")
    )

    for step in range(60):

        time_idx_value = int(
            decoder_time_idx[i, step]
        )

        row_info = series_dates.loc[
            time_idx_value
        ]

        forecast_rows.append({
            "series_id": series_id,
            "store_nbr": row_info["store_nbr"],
            "item_nbr": row_info["item_nbr"],
            "date": row_info["date"],
            "p10": float(p10[i, step]),
            "p50": float(p50[i, step]),
            "p90": float(p90[i, step]),
        })

tft_forecast_60 = pd.DataFrame(
    forecast_rows
)

print("Forecast rows:", len(tft_forecast_60))

print(
    "Forecast range:",
    tft_forecast_60["date"].min(),
    "to",
    tft_forecast_60["date"].max()
)

print(
    "Series:",
    tft_forecast_60["series_id"].nunique()
)

tft_forecast_60.head()

Forecast rows: 18000
Forecast range: 2017-05-01 00:00:00 to 2017-06-29 00:00:00
Series: 300


,series_id,store_nbr,item_nbr,date,p10,p50,p90
0,10_1080001,10,1080001,2017-05-01,6.488178e-01,4.136733e+00,1.049434e+01
1,10_1080001,10,1080001,2017-05-02,3.773254e-23,8.872295e-23,6.514126e-24
2,10_1080001,10,1080001,2017-05-03,1.728430e-25,7.686555e-26,4.782748e-27
3,10_1080001,10,1080001,2017-05-04,2.494859e-26,6.624704e-27,3.400855e-28
4,10_1080001,10,1080001,2017-05-05,2.744303e-26,7.544982e-28,7.564513e-30


In [20]:
# ============================================================
# Cell 16B — Fix quantile crossing for uncertainty intervals
# ============================================================

# Keep the original model outputs untouched.
tft_forecast_60["p10_raw"] = tft_forecast_60["p10"]
tft_forecast_60["p50_raw"] = tft_forecast_60["p50"]
tft_forecast_60["p90_raw"] = tft_forecast_60["p90"]

# Count rows where the quantiles are not ordered correctly.
crossing_mask = (
    (tft_forecast_60["p10"] > tft_forecast_60["p50"])
    | (tft_forecast_60["p50"] > tft_forecast_60["p90"])
)

print("Quantile-crossing rows:", crossing_mask.sum())
print(
    "Percentage:",
    round(crossing_mask.mean() * 100, 2),
    "%"
)

# ------------------------------------------------------------
# Monotonic rearrangement
# ------------------------------------------------------------
# Sort the three quantile predictions within each row.
# This guarantees:
#
# P10 <= P50 <= P90
#
# We will use these adjusted values later for uncertainty bands.
quantile_values = tft_forecast_60[
    ["p10_raw", "p50_raw", "p90_raw"]
].to_numpy()

sorted_quantiles = np.sort(
    quantile_values,
    axis=1
)

tft_forecast_60["p10_adjusted"] = sorted_quantiles[:, 0]
tft_forecast_60["p50_adjusted"] = sorted_quantiles[:, 1]
tft_forecast_60["p90_adjusted"] = sorted_quantiles[:, 2]

print(
    "Remaining crossings:",
    (
        (tft_forecast_60["p10_adjusted"] >
         tft_forecast_60["p50_adjusted"])
        |
        (tft_forecast_60["p50_adjusted"] >
         tft_forecast_60["p90_adjusted"])
    ).sum()
)

Quantile-crossing rows: 2875
Percentage: 15.97 %
Remaining crossings: 0


In [21]:
# ============================================================
# Cell 17 — First 30 days of TFT rolling experiment
# ============================================================

FIRST_WINDOW_END = pd.Timestamp("2017-05-30")

tft_may_forecast = tft_forecast_60[
    tft_forecast_60["date"].between(
        FIRST_FORECAST_START,
        FIRST_WINDOW_END
    )
].copy()

# Use the ORIGINAL TFT P50 for model evaluation.
tft_may_forecast["prediction"] = np.clip(
    tft_may_forecast["p50_raw"],
    0,
    None
)

print("First-window rows:", len(tft_may_forecast))

print(
    "Date range:",
    tft_may_forecast["date"].min(),
    "to",
    tft_may_forecast["date"].max()
)

print("Series:", tft_may_forecast["series_id"].nunique())

First-window rows: 9000
Date range: 2017-05-01 00:00:00 to 2017-05-30 00:00:00
Series: 300


In [22]:
# ============================================================
# Cell 18 — TFT reforecast using ACTUAL May history
# ============================================================

# We need a fresh 60-day decoder beginning May 31.
#
# Encoder ends:      May 30
# Decoder begins:    May 31
# 60-day decoder ends: July 29
#
# We will later use only the FIRST 30 decoder days
# (May 31 → June 29) for your 30+30 experiment.

SECOND_TFT_CUTOFF = pd.Timestamp("2017-07-29")

second_tft_data = tft_data[
    tft_data["date"] <= SECOND_TFT_CUTOFF
].copy()

# Reuse the exact encoders/scalers/configuration learned during training.
second_tft_dataset = TimeSeriesDataSet.from_parameters(
    tft_dataset_parameters,
    second_tft_data,
    predict=True,
    stop_randomization=True,
)

second_tft_loader = second_tft_dataset.to_dataloader(
    train=False,
    batch_size=256,
    num_workers=0,
)

print("Prediction sequences:", len(second_tft_dataset))
print("Prediction batches:", len(second_tft_loader))

# ------------------------------------------------------------
# Generate the refreshed forecast
# ------------------------------------------------------------

with torch.no_grad():
    second_tft_output = tft_model.predict(
        second_tft_loader,
        mode="quantiles",
        return_x=True
    )

second_quantiles = (
    second_tft_output.output
    .cpu()
    .numpy()
)

print("Prediction shape:", second_quantiles.shape)

print(
    "First decoder time_idx:",
    int(
        second_tft_output.x["decoder_time_idx"][0, 0]
    )
)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


Prediction sequences: 300
Prediction batches: 2
Prediction shape: (300, 60, 3)
First decoder time_idx: 1610


In [23]:
# ============================================================
# Cell 19 — Build refreshed TFT forecast table
# ============================================================

second_group_codes = (
    second_tft_output.x["groups"]
    .squeeze(1)
    .cpu()
    .numpy()
)

second_decoder_time_idx = (
    second_tft_output.x["decoder_time_idx"]
    .cpu()
    .numpy()
)

second_p10 = second_quantiles[:, :, 0]
second_p50 = second_quantiles[:, :, 1]
second_p90 = second_quantiles[:, :, 2]

second_rows = []

for i, group_code in enumerate(second_group_codes):

    series_id = group_code_to_series[int(group_code)]

    series_dates = (
        tft_data[
            tft_data["series_id"] == series_id
        ][["time_idx", "date", "store_nbr", "item_nbr"]]
        .drop_duplicates()
        .set_index("time_idx")
    )

    for step in range(60):

        time_idx_value = int(
            second_decoder_time_idx[i, step]
        )

        row_info = series_dates.loc[
            time_idx_value
        ]

        second_rows.append({
            "series_id": series_id,
            "store_nbr": row_info["store_nbr"],
            "item_nbr": row_info["item_nbr"],
            "date": row_info["date"],
            "p10_raw": float(second_p10[i, step]),
            "p50_raw": float(second_p50[i, step]),
            "p90_raw": float(second_p90[i, step]),
        })

tft_second_forecast_60 = pd.DataFrame(second_rows)

print("Forecast rows:", len(tft_second_forecast_60))

print(
    "Forecast range:",
    tft_second_forecast_60["date"].min(),
    "to",
    tft_second_forecast_60["date"].max()
)

print("Series:", tft_second_forecast_60["series_id"].nunique())

Forecast rows: 18000
Forecast range: 2017-05-31 00:00:00 to 2017-07-29 00:00:00
Series: 300


In [24]:
# ============================================================
# Cell 20 — Evaluate TFT rolling 30 + 30 strategy
# ============================================================

SECOND_WINDOW_END = pd.Timestamp("2017-06-29")

# Keep only the first 30 days from the refreshed forecast.
tft_june_forecast = tft_second_forecast_60[
    tft_second_forecast_60["date"].between(
        SECOND_FORECAST_START,
        SECOND_WINDOW_END
    )
].copy()

# Use ORIGINAL P50 for honest point-forecast evaluation.
tft_june_forecast["prediction"] = np.clip(
    tft_june_forecast["p50_raw"],
    0,
    None
)

print("Second-window rows:", len(tft_june_forecast))
print(
    "Second-window range:",
    tft_june_forecast["date"].min(),
    "to",
    tft_june_forecast["date"].max()
)

# ------------------------------------------------------------
# Combine May forecast + refreshed June forecast
# ------------------------------------------------------------

tft_rolling_forecast = pd.concat(
    [
        tft_may_forecast[
            ["series_id", "store_nbr", "item_nbr", "date", "prediction"]
        ],
        tft_june_forecast[
            ["series_id", "store_nbr", "item_nbr", "date", "prediction"]
        ],
    ],
    ignore_index=True
)

# Attach actual demand.
tft_actuals = tft_data[
    [
        "series_id",
        "store_nbr",
        "item_nbr",
        "date",
        "demand_target",
    ]
].rename(
    columns={"demand_target": "actual"}
)

tft_rolling_eval = tft_rolling_forecast.merge(
    tft_actuals,
    on=[
        "series_id",
        "store_nbr",
        "item_nbr",
        "date",
    ],
    how="left",
)

# Unknown targets are not evaluated.
tft_rolling_eval = tft_rolling_eval[
    tft_rolling_eval["actual"].notna()
].copy()

# ------------------------------------------------------------
# MAE
# ------------------------------------------------------------

tft_rolling_mae = np.mean(
    np.abs(
        tft_rolling_eval["actual"]
        - tft_rolling_eval["prediction"]
    )
)

# ------------------------------------------------------------
# WAPE
# ------------------------------------------------------------

tft_rolling_wape = (
    np.abs(
        tft_rolling_eval["actual"]
        - tft_rolling_eval["prediction"]
    ).sum()
    /
    np.abs(
        tft_rolling_eval["actual"]
    ).sum()
    * 100
)

print("\nRows evaluated:", len(tft_rolling_eval))

print(
    "TFT rolling 30+30 MAE:",
    round(tft_rolling_mae, 4)
)

print(
    "TFT rolling 30+30 WAPE:",
    round(tft_rolling_wape, 4),
    "%"
)

print("\nLightGBM rolling 30+30:")
print("MAE:  3.1043")
print("WAPE: 42.6187 %")

Second-window rows: 9000
Second-window range: 2017-05-31 00:00:00 to 2017-06-29 00:00:00

Rows evaluated: 17998
TFT rolling 30+30 MAE: 3.0616
TFT rolling 30+30 WAPE: 42.0331 %

LightGBM rolling 30+30:
MAE:  3.1043
WAPE: 42.6187 %


In [25]:
# ============================================================
# Cell 21 — TFT fixed-horizon evaluation
# ============================================================

# Use ORIGINAL P50 as the point forecast.
fixed_tft = tft_forecast_60.copy()

fixed_tft["prediction"] = np.clip(
    fixed_tft["p50_raw"],
    0,
    None
)

# Attach actual demand.
actuals = tft_data[
    [
        "series_id",
        "store_nbr",
        "item_nbr",
        "date",
        "demand_target",
    ]
].rename(
    columns={"demand_target": "actual"}
)

fixed_tft_eval = fixed_tft.merge(
    actuals,
    on=[
        "series_id",
        "store_nbr",
        "item_nbr",
        "date",
    ],
    how="left",
)

# Keep only rows with known actual demand.
fixed_tft_eval = fixed_tft_eval[
    fixed_tft_eval["actual"].notna()
].copy()

# Day number within forecast horizon:
# May 1 = 1, May 2 = 2, ..., Jun 29 = 60
fixed_tft_eval["forecast_day"] = (
    fixed_tft_eval["date"]
    - pd.Timestamp("2017-05-01")
).dt.days + 1

HORIZONS = [7, 14, 30, 60]

results = []

for horizon in HORIZONS:

    subset = fixed_tft_eval[
        fixed_tft_eval["forecast_day"] <= horizon
    ].copy()

    mae = np.mean(
        np.abs(
            subset["actual"]
            - subset["prediction"]
        )
    )

    wape = (
        np.abs(
            subset["actual"]
            - subset["prediction"]
        ).sum()
        /
        np.abs(
            subset["actual"]
        ).sum()
        * 100
    )

    results.append({
        "horizon_days": horizon,
        "rows": len(subset),
        "MAE": mae,
        "WAPE": wape,
    })

tft_horizon_results = pd.DataFrame(results)

print(tft_horizon_results)

   horizon_days   rows       MAE       WAPE
0             7   2100  3.746378  43.229765
1            14   4200  3.329033  42.750914
2            30   9000  3.167008  43.000511
3            60  17998  3.097539  42.526352
